# ShortsLM — Personal YouTube Shorts & Instagram Reels Feature Extraction

**Resumable Google Colab notebook with anonymous downloads, ASR fallback, OCR QA and feature bundle v3.0.0.**

This revision targets the reported Colab exception:
`ConvertPirAttribute2RuntimeAttribute not support pir::ArrayAttribute<pir::DoubleAttribute>`

Changes:
- Installs PaddlePaddle CPU 3.2.2, with a CPython 3.13 Linux wheel.
- Disables PaddleOCR MKL-DNN acceleration to avoid the oneDNN/PIR path in the reported traceback.
- Executes a real synthetic-image OCR preflight before starting the URL queue.
- Requeues previously pushed JSON records whose OCR stage failed, while preserving records with successful OCR.
- Keeps the persistent Drive checkpoint, anonymous repository downloader, media cleanup, and per-video GitHub commits.

Anonymous public-media access remains subject to platform availability and rate limits.


### Feature extraction update (v3.0.0)
- Whisper retries once without VAD if its first pass returns zero segments; decoding diagnostics, script hints and retry state are retained.
- OCR preserves raw candidates and adds provisional confidence review flags, normalized boxes and adjacent-frame text track IDs.
- Each analysis includes modality-separated `feature_bundle`, temporal events, quality report and extractor/config/rights provenance.
- P1/P2 person, action, object, scene, music-ID and narrative models remain explicitly `NOT_YET_PROCESSED` until licensing, resource and human-reviewed validation gates pass. No such predictions are fabricated.
- Older saved records without bundle v3.0.0 are automatically reprocessed and refreshed in the private feature repository.
- OCR confidence threshold is provisional, not calibrated; review a labelled pilot before training on OCR-derived labels.


- Every v3 bundle marks `training_readiness=NOT_READY` until item-level training permission, human-reviewed labels, deduplication and leakage-safe dataset splits are verified.


## 1. Mount Drive and create persistent workspace

In [1]:
from google.colab import drive, files
drive.mount("/content/drive")

from pathlib import Path
import os
import sys
import re
import json
import time
import shutil
import tempfile
import subprocess
import base64
import hashlib
import importlib
from datetime import datetime, timezone
from urllib.parse import quote
import requests

DRIVE_ROOT = Path("/content/drive/MyDrive/ShortsLM_Personal_Pilot")
INPUT_DIR = DRIVE_ROOT / "input"
STATE_DIR = DRIVE_ROOT / "state"
FEATURES_DIR = DRIVE_ROOT / "features"
SECRETS_DIR = DRIVE_ROOT / "secrets"

for folder in (INPUT_DIR, STATE_DIR, FEATURES_DIR, SECRETS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

COMPLETED_FILE = STATE_DIR / "completed.txt"
REMAINING_FILE = STATE_DIR / "remaining.txt"
FAILED_LOG = STATE_DIR / "failed.jsonl"
PUSH_LOG = STATE_DIR / "github_push_log.jsonl"

print("Persistent workspace:", DRIVE_ROOT)
print("Input folder:", INPUT_DIR)
print("Feature folder:", FEATURES_DIR)


Mounted at /content/drive
Persistent workspace: /content/drive/MyDrive/ShortsLM_Personal_Pilot
Input folder: /content/drive/MyDrive/ShortsLM_Personal_Pilot/input
Feature folder: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features


## 2. Put your URL TXT files in Drive

In [2]:
from google.colab import files

def find_input_file(candidates):
    wanted = {name.casefold() for name in candidates}
    for path in INPUT_DIR.iterdir():
        if path.is_file() and path.name.casefold() in wanted:
            return path
    return None

ig_file = find_input_file([
    "Instagram_reels.txt", "instagram_reels.txt", "instagram reels.txt"
])
yt_file = find_input_file([
    "YouTube_shorts.txt", "youtube_shorts.txt",
    "YouTube_shorts_YouTube_shorts.txt", "youtube_shorts_youtube_shorts.txt"
])

missing = []
if ig_file is None:
    missing.append("Instagram_reels.txt")
if yt_file is None:
    missing.append("YouTube_shorts.txt")

if missing:
    print("Upload the missing TXT file(s):", ", ".join(missing))
    uploaded = files.upload()
    for filename, data in uploaded.items():
        (INPUT_DIR / filename).write_bytes(data)

ig_file = find_input_file([
    "Instagram_reels.txt", "instagram_reels.txt", "instagram reels.txt"
])
yt_file = find_input_file([
    "YouTube_shorts.txt", "youtube_shorts.txt",
    "YouTube_shorts_YouTube_shorts.txt", "youtube_shorts_youtube_shorts.txt"
])

if ig_file is None and yt_file is None:
    raise FileNotFoundError(
        f"No URL TXT files found in {INPUT_DIR}. Upload at least one file."
    )

print("Instagram URL file:", ig_file)
print("YouTube URL file:", yt_file)


Instagram URL file: /content/drive/MyDrive/ShortsLM_Personal_Pilot/input/instagram_reels.txt
YouTube URL file: /content/drive/MyDrive/ShortsLM_Personal_Pilot/input/youtube_shorts.txt


## 3. Clone ShortsLM and install its real dependencies

In [3]:
# ================================================================
# SHORTSLM CHECKOUT + RUNTIME DEPENDENCIES
# PyAV < 19 for faster-whisper; PaddlePaddle CPU for PaddleOCR.
# ================================================================

REPO_URL = "https://github.com/Arhanpg/Shorts-Language-Model.git"
REPO_BRANCH = "feat/p0-quality-feature-bundle"
PROJECT_DIR = Path("/content/Shorts-Language-Model")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-e",
        ".[audio,ocr,discovery,legacy-discovery,instagram-fallback]",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

# Enforce the PyAV compatibility pin after all extras are installed.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps", "av>=11,<19",
    ],
    check=True,
)

# PaddleOCR is only the OCR toolkit; the PaddlePaddle inference engine
# is a separate dependency. Install CPU PaddlePaddle for Colab reliability.
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
paddle_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "paddlepaddle==3.2.2"],
    capture_output=True,
    text=True,
    check=False,
)

if paddle_install.returncode != 0:
    print("PaddlePaddle installation did not complete:")
    print((paddle_install.stderr or paddle_install.stdout)[-1800:])

import av
AV_VERSION = av.__version__
AV_MAJOR = int(AV_VERSION.split(".")[0])
if AV_MAJOR >= 19:
    raise RuntimeError(
        f"Incompatible PyAV {AV_VERSION}. A version below 19 is required by "
        "the installed faster-whisper path. Restart runtime and rerun setup."
    )
print("PyAV:", AV_VERSION)

PADDLE_READY = False
PADDLE_VERSION = None
try:
    import paddle
    paddle.utils.run_check()
    PADDLE_VERSION = paddle.__version__
    if PADDLE_VERSION != "3.2.2":
        raise RuntimeError(f"Expected PaddlePaddle 3.2.2; found {PADDLE_VERSION}")
    PADDLE_READY = True
except Exception as exc:
    print("PaddlePaddle runtime check failed:", type(exc).__name__, str(exc)[:800])

ENABLE_OCR = PADDLE_READY
print("PaddlePaddle:", PADDLE_VERSION if PADDLE_READY else "unavailable")
print("OCR enabled:", ENABLE_OCR)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "nodejs"], check=True)
print("Repository installed:", PROJECT_DIR)
print("Node:", shutil.which("node"))
print("FFmpeg:", shutil.which("ffmpeg"))


PyAV: 18.1.1


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 
PaddlePaddle works well on 1 CPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
PaddlePaddle: 3.2.2
OCR enabled: True


/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


Repository installed: /content/Shorts-Language-Model
Node: /usr/bin/node
FFmpeg: /usr/bin/ffmpeg


## 4. Load ShortsLM analyzer and configure feature extraction

In [4]:
# ================================================================
# SHORTSLM ANALYZER CONFIGURATION
# ================================================================

import torch
import cv2

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import shorts_intelligence.telemetry.ocr as ocr_module
importlib.reload(ocr_module)

import shorts_intelligence.analyzer as analyzer_module
importlib.reload(analyzer_module)

from shorts_intelligence.config import AnalyzerConfig
from shorts_intelligence.discovery.models import DiscoveredCandidate
from shorts_intelligence.schemas import FrameEvidence
from shorts_intelligence.telemetry.ocr import run_ocr

analyze_video = analyzer_module.analyze_video

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ASR_MODEL = "small"
ASR_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_DURATION_SECONDS = 180.0
MAX_FILE_SIZE_MB = 1000
MAX_KEYFRAMES = 24
MAX_OCR_FRAMES = 12

SESSION_BUDGET_SECONDS = 4 * 60 * 60 + 30 * 60
SESSION_STARTED = time.monotonic()

ANALYZER_CONFIG = AnalyzerConfig(
    max_file_size_mb=MAX_FILE_SIZE_MB,
    max_duration_seconds=MAX_DURATION_SECONDS,
    max_keyframes=MAX_KEYFRAMES,
    max_ocr_frames=MAX_OCR_FRAMES,
    asr_model=ASR_MODEL,
    asr_device=ASR_DEVICE,
    asr_timeout_seconds=300,
    enable_asr=True,
    enable_ocr=ENABLE_OCR,
    enable_audio_analysis=True,
    keep_artifacts=False,
)

print("ASR:", ASR_MODEL, "| device:", ASR_DEVICE)
print("OCR:", "enabled" if ENABLE_OCR else "disabled; PaddlePaddle unavailable")

# Real synthetic-image inference is required before starting the URL queue.
if not ENABLE_OCR:
    raise RuntimeError(
        "PaddlePaddle 3.2.2 did not pass runtime verification. "
        "Do not start feature extraction."
    )

import numpy as np

_preflight_image = np.full((180, 640, 3), 255, dtype=np.uint8)
cv2.putText(
    _preflight_image, "ShortsLM OCR preflight 123", (18, 100),
    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2, cv2.LINE_AA,
)
_preflight_evidence = FrameEvidence(
    timestamp_seconds=0.0, frame_index=0,
    reason=["ocr_preflight"], motion_score=0.0,
)

try:
    _preflight_spans, OCR_PREFLIGHT_ENGINE = run_ocr(
        [(_preflight_evidence, _preflight_image)],
        max_frames=1,
        prefer_gpu=False,
    )
except Exception as exc:
    ENABLE_OCR = False
    raise RuntimeError(
        "PaddleOCR preflight inference failed. The video queue has NOT been "
        "started. Underlying error: "
        f"{type(exc).__name__}: {exc}"
    ) from exc

print(
    "OCR preflight passed:",
    OCR_PREFLIGHT_ENGINE,
    "| detected spans:",
    len(_preflight_spans),
)



CUDA available: False
ASR: small | device: cpu
OCR: enabled


Creating model: ('PP-OCRv6_medium_det', None, None)
Connectivity check to the model hoster has been skipped because `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` is enabled.
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

OCR preflight passed: paddleocr-3-cpu | detected spans: 1


## 5. Anonymous public-media downloads (no login-session setup)

The repository downloader attempts public media anonymously. Explicit platform challenges and rate limits stop fallback attempts.


## 6. Parse and normalize the URL lists


In [5]:
YT_RE = re.compile(
    r"(?:youtu\.be/|youtube\.com/(?:shorts/|watch\?v=|embed/|live/))([A-Za-z0-9_-]{11})",
    re.I
)
IG_RE = re.compile(
    r"instagram\.com/(?:reel|reels|p|tv)/([A-Za-z0-9_-]+)",
    re.I
)

def read_source_file(path):
    if path is None:
        return []
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

def normalize_url(raw_url, platform):
    raw_url = raw_url.strip()
    if platform == "youtube":
        match = YT_RE.search(raw_url)
        if not match:
            raise ValueError(f"Unrecognized YouTube URL: {raw_url}")
        source_id = match.group(1)
        return source_id, f"https://www.youtube.com/shorts/{source_id}"

    match = IG_RE.search(raw_url)
    if not match:
        raise ValueError(f"Unrecognized Instagram URL: {raw_url}")
    source_id = match.group(1)
    return source_id, f"https://www.instagram.com/reel/{source_id}/"

items = []
seen = set()
invalid_urls = []

for platform, path in (("instagram", ig_file), ("youtube", yt_file)):
    for raw in read_source_file(path):
        try:
            source_id, canonical_url = normalize_url(raw, platform)
            key = (platform, source_id)
            if key not in seen:
                seen.add(key)
                items.append({
                    "platform": platform,
                    "source_id": source_id,
                    "url": canonical_url,
                })
        except Exception as exc:
            invalid_urls.append({"url": raw, "platform": platform, "error": str(exc)})

if not items:
    raise RuntimeError("No valid video URLs were found in the supplied TXT files.")

source_urls = [item["url"] for item in items]
ITEM_BY_URL = {item["url"]: item for item in items}

print("Unique valid URLs:", len(items))
print("YouTube:", sum(x["platform"] == "youtube" for x in items))
print("Instagram:", sum(x["platform"] == "instagram" for x in items))
print("Invalid URL lines:", len(invalid_urls))
for bad in invalid_urls[:10]:
    print("INVALID:", bad["url"], "|", bad["error"])


Unique valid URLs: 310871
YouTube: 118023
Instagram: 192848
Invalid URL lines: 0


## 7. Initialize resumable state


In [6]:
def read_lines(path):
    if not path.exists():
        return []
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]

def write_lines_atomic(path, values):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        "".join(str(value) + "\n" for value in values),
        encoding="utf-8"
    )
    temporary.replace(path)

def append_jsonl(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(payload, ensure_ascii=False) + "\n")
        stream.flush()

def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8"
    )
    temporary.replace(path)

completed_set = set(read_lines(COMPLETED_FILE))
# Ignore stale/foreign URLs in an old remaining file; source TXT files are authoritative.
remaining_set = set(read_lines(REMAINING_FILE))
pending_urls = [
    url for url in source_urls
    if url not in completed_set
]
write_lines_atomic(REMAINING_FILE, pending_urls)
write_lines_atomic(COMPLETED_FILE, sorted(completed_set))

print("Completed:", len(completed_set))
print("Remaining:", len(pending_urls))
print("Checkpoint files:", COMPLETED_FILE, REMAINING_FILE)


Completed: 18
Remaining: 310853
Checkpoint files: /content/drive/MyDrive/ShortsLM_Personal_Pilot/state/completed.txt /content/drive/MyDrive/ShortsLM_Personal_Pilot/state/remaining.txt


## 9. Configure the private GitHub destination

Feature files are pushed to the existing private repository `Arhanpg/ShortsLM-Personal-Features`, branch `main`. Because the repository is private, Colab must authenticate with a token that has access to this exact repository and **Contents: Read and write** permission. GitHub may return 404 when a token cannot see a private repository; that does not mean the repository should be made public.

Add a NEW fine-grained token as Google Colab Secret `GH_TOKEN`. Revoke the token previously pasted into the conversation. No credential is embedded in this notebook.

In [7]:
# ================================================================
# PRIVATE GITHUB FEATURE DESTINATION
# ================================================================

GITHUB_REPO = "Arhanpg/ShortsLM-Personal-Features"
GITHUB_BRANCH = "main"
GITHUB_FEATURES_PREFIX = "personal_pilot/features"

from google.colab import userdata
import getpass

try:
    GH_TOKEN = userdata.get("GH_TOKEN")
except Exception:
    GH_TOKEN = None

if not GH_TOKEN:
    GH_TOKEN = getpass.getpass("New GitHub token (hidden): ").strip()
if not GH_TOKEN:
    raise RuntimeError("Missing GH_TOKEN. Add it to Colab Secrets.")

GH_API = "https://api.github.com"
GH_HEADERS = {
    "Authorization": f"Bearer {GH_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

repo_response = requests.get(
    f"{GH_API}/repos/{GITHUB_REPO}",
    headers=GH_HEADERS,
    timeout=30,
)

if repo_response.status_code == 404:
    raise RuntimeError(
        f"Cannot access {GITHUB_REPO}. Create it as PRIVATE and grant GH_TOKEN access."
    )
if repo_response.status_code in (401, 403):
    raise RuntimeError(
        f"GitHub authentication/permission failure: HTTP {repo_response.status_code}. "
        "Check the replacement token and Contents read/write permission."
    )
repo_response.raise_for_status()

repo_info = repo_response.json()
if repo_info.get("private") is not True:
    raise RuntimeError("Feature-output repository must be PRIVATE.")

branch_response = requests.get(
    f"{GH_API}/repos/{GITHUB_REPO}/branches/{GITHUB_BRANCH}",
    headers=GH_HEADERS,
    timeout=30,
)
branch_response.raise_for_status()
print("Verified private GitHub destination:", GITHUB_REPO, GITHUB_BRANCH)


Verified private GitHub destination: Arhanpg/ShortsLM-Personal-Features main


## 10. GitHub commit helper

In [8]:
def github_commit_json(path_in_repo, payload, message):
    encoded_path = quote(path_in_repo, safe="/")
    api_url = f"{GH_API}/repos/{GITHUB_REPO}/contents/{encoded_path}"

    raw_json = (
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    ).encode("utf-8")

    body = {
        "message": message,
        "content": base64.b64encode(raw_json).decode("ascii"),
        "branch": GITHUB_BRANCH,
    }

    existing = requests.get(
        api_url,
        headers=GH_HEADERS,
        params={"ref": GITHUB_BRANCH},
        timeout=30
    )

    if existing.status_code == 200:
        body["sha"] = existing.json()["sha"]
    elif existing.status_code != 404:
        raise RuntimeError(
            f"GitHub file lookup failed: HTTP {existing.status_code}: "
            f"{existing.text[:500]}"
        )

    response = requests.put(
        api_url,
        headers=GH_HEADERS,
        json=body,
        timeout=90
    )

    if response.status_code not in (200, 201):
        raise RuntimeError(
            f"GitHub feature commit failed: HTTP {response.status_code}: "
            f"{response.text[:700]}"
        )

    result = response.json()
    return (
        result.get("commit", {}).get("sha"),
        result.get("content", {}).get("html_url"),
    )


## 8. Download through ShortsLM's repository adapters


In [9]:
# ================================================================
# SHORTSLM REPOSITORY-NATIVE ANONYMOUS DOWNLOADER
# ================================================================

from pathlib import Path
from shorts_intelligence.discovery.downloader import download_candidate, DownloadError
from shorts_intelligence.discovery.models import DiscoveredCandidate

def download_video(item, output_dir):
    """Use the repository public-media downloader without sessions/cookies."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    candidate = DiscoveredCandidate(
        platform=item["platform"],
        source_id=item["source_id"],
        canonical_url=item["url"],
        short_format_status="unknown",
        status="metric_unknown",
        metadata={
            "source": "personal_url_txt",
            "download_mode": "anonymous",
            "authentication_session_supplied": False,
        },
    )

    print(f"ShortsLM downloader | {candidate.platform} | {candidate.source_id}")
    media_path = download_candidate(candidate, output_dir)
    media_path = Path(media_path).resolve(strict=True)

    if not media_path.is_file() or media_path.stat().st_size <= 0:
        raise DownloadError("Repository downloader returned an empty or missing file.")

    print(
        "Download successful:",
        media_path.name,
        "|",
        round(media_path.stat().st_size / (1024 * 1024), 2),
        "MB",
    )
    return media_path

print("Anonymous repository-native downloader ready.")
print("No Instagram session or YouTube cookie required.")


Anonymous repository-native downloader ready.
No Instagram session or YouTube cookie required.


## 11. Process one URL: download → analyze → persist → delete → push


In [10]:
def safe_id(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value))[:120]

def process_one(item):
    platform = item["platform"]
    source_id = item["source_id"]
    url = item["url"]

    local_json = FEATURES_DIR / platform / f"{safe_id(source_id)}.json"
    github_path = (
        f"{GITHUB_FEATURES_PREFIX}/{platform}/{safe_id(source_id)}.json"
    )

    # Reuse valid saved JSON. Re-download/re-analyze records whose prior
    # OCR stage failed; the old run did not persist frame images for OCR retry.
    if local_json.exists():
        existing_payload = json.loads(local_json.read_text(encoding="utf-8"))
        existing_analysis = existing_payload.get("analysis", {})
        existing_ocr_status = existing_analysis.get("stages", {}).get("ocr", {}).get("status")
        existing_bundle_version = (
            existing_analysis.get("feature_bundle", {}).get("bundle_version")
        )
        needs_refresh = (
            existing_ocr_status == "failed"
            or existing_bundle_version != "3.0.0"
        )
        if needs_refresh:
            print(
                "Existing record requires feature bundle v3.0.0 / QA refresh; "
                "re-downloading and reprocessing:", local_json,
            )
            local_json.unlink()
        else:
            payload = existing_payload
            print("Reusing saved feature JSON:", local_json)

    if not local_json.exists():
        temp_dir = Path(tempfile.mkdtemp(
            prefix=f"shortslm_{platform}_{safe_id(source_id)}_"
        ))
        try:
            media_path = download_video(item, temp_dir)
            media_path = Path(media_path).resolve(strict=True)

            if not media_path.is_file() or media_path.stat().st_size == 0:
                raise RuntimeError("Downloader returned an empty or missing media file.")

            if media_path.stat().st_size > MAX_FILE_SIZE_MB * 1024 * 1024:
                raise RuntimeError(
                    f"Downloaded media exceeds {MAX_FILE_SIZE_MB} MB limit."
                )

            print("Downloaded:", media_path.name, "|", round(media_path.stat().st_size / 1e6, 2), "MB")
            report = analyze_video(
                media_path,
                temp_dir / "analysis.json",
                config=ANALYZER_CONFIG
            )

            payload = {
                "schema_version": "shortslm.personal-pilot.v2",
                "record_type": "short_video_feature_observation",
                "platform": platform,
                "source_id": source_id,
                "canonical_url": url,
                "processed_at": datetime.now(timezone.utc).isoformat(),
                "pipeline": {
                    "repository": REPO_URL,
                    "repository_branch": REPO_BRANCH,
                    "asr_model": ASR_MODEL,
                    "asr_device": ASR_DEVICE,
                    "max_duration_seconds": MAX_DURATION_SECONDS,
                    "max_keyframes": MAX_KEYFRAMES,
                    "max_ocr_frames": MAX_OCR_FRAMES,
                    "analyzer_config": "repository AnalyzerConfig",
                },
                "analysis": report.model_dump(mode="json"),
            }

            # Persist features to Drive before deleting the downloaded video.
            atomic_json(local_json, payload)
            print("Feature JSON saved to Drive:", local_json)

        finally:
            # Always remove downloaded video, sidecars, and temporary analysis files.
            shutil.rmtree(temp_dir, ignore_errors=True)
            print("Temporary media directory removed.")

    commit_sha, html_url = github_commit_json(
        github_path,
        payload,
        f"Add ShortsLM personal features for {platform}:{source_id}"
    )

    return {
        "platform": platform,
        "source_id": source_id,
        "url": url,
        "local_json": str(local_json),
        "github_path": github_path,
        "commit_sha": commit_sha,
        "github_url": html_url,
    }


## 12. Run the sequential, resumable feature extraction pipeline


In [ ]:
# ================================================================
# SHORTSLM PERSONAL FEATURE EXTRACTION — RESUMABLE RUNNER
# ================================================================

successes = 0
failures = 0
completed_set = set(read_lines(COMPLETED_FILE))

# Requeue any previously pushed records with failed OCR, so the corrected
# analyzer redownloads and reprocesses them. Successfully completed records
# are preserved and will not be downloaded again.
ocr_retry_urls = []
for candidate_url in source_urls:
    if candidate_url not in completed_set:
        continue
    candidate_item = ITEM_BY_URL.get(candidate_url)
    if not candidate_item:
        continue
    old_feature = (
        FEATURES_DIR
        / candidate_item["platform"]
        / f"{safe_id(candidate_item['source_id'])}.json"
    )
    if not old_feature.exists():
        continue
    try:
        old_payload = json.loads(old_feature.read_text(encoding="utf-8"))
        old_analysis = old_payload.get("analysis", {})
        old_status = old_analysis.get("stages", {}).get("ocr", {}).get("status")
        old_bundle_version = old_analysis.get("feature_bundle", {}).get("bundle_version")
        if old_status == "failed" or old_bundle_version != "3.0.0":
            ocr_retry_urls.append(candidate_url)
    except (OSError, ValueError, TypeError):
        continue

completed_set.difference_update(ocr_retry_urls)
queue = [url for url in source_urls if url not in completed_set]
write_lines_atomic(COMPLETED_FILE, sorted(completed_set))
write_lines_atomic(REMAINING_FILE, queue)

print("=" * 70)
print("SHORTSLM PERSONAL FEATURE EXTRACTION")
print("=" * 70)
print("Source URLs:", f"{len(source_urls):,}")
print("Already completed (valid records):", f"{len(completed_set):,}")
print("Previously completed records requiring v3.0.0 / QA refresh; requeued:", f"{len(ocr_retry_urls):,}")
print("To process:", f"{len(queue):,}")
if len(source_urls) > 10000:
    print("NOTICE: More than 10,000 source URLs detected. No URL cap is applied; the session time checkpoint remains active.")
print("ASR: enabled | PyAV:", AV_VERSION)
print("OCR:", "enabled" if ENABLE_OCR else "disabled (PaddlePaddle unavailable)")
print("Download mode: anonymous")
print("=" * 70)

try:
    while queue:
        if time.monotonic() - SESSION_STARTED >= SESSION_BUDGET_SECONDS:
            print("Colab safety time budget reached. Remaining URLs checkpointed.")
            break

        url = queue[0]
        item = ITEM_BY_URL.get(url)

        if item is None:
            failures += 1
            append_jsonl(FAILED_LOG, {
                "url": url,
                "error": "URL missing from ITEM_BY_URL",
                "at": datetime.now(timezone.utc).isoformat(),
            })
            queue.pop(0)
            write_lines_atomic(REMAINING_FILE, queue)
            continue

        print(
            f"\n[{len(completed_set) + 1}/{len(source_urls)}] "
            f"{item['platform']} | {item['source_id']} | remaining: {len(queue):,}"
        )

        try:
            result = process_one(item)

            # Show failed optional stages; the analyzer still saves a partial report.
            feature_file = Path(result["local_json"])
            feature_payload = json.loads(feature_file.read_text(encoding="utf-8"))
            stages = feature_payload.get("analysis", {}).get("stages", {})
            failed_stages = [
                name for name, stage in stages.items()
                if isinstance(stage, dict) and stage.get("status") == "failed"
            ]
            if failed_stages:
                print("WARNING: feature record has failed stages:", failed_stages)

            completed_set.add(url)
            append_jsonl(PUSH_LOG, {
                **result,
                "pushed_at": datetime.now(timezone.utc).isoformat(),
            })

            queue.pop(0)
            write_lines_atomic(COMPLETED_FILE, sorted(completed_set))
            write_lines_atomic(REMAINING_FILE, queue)

            successes += 1
            print("PUSHED:", result["github_url"])
            print("COMMIT:", result["commit_sha"])

        except Exception as exc:
            failures += 1
            error_text = f"{type(exc).__name__}: {exc}"
            append_jsonl(FAILED_LOG, {
                "url": url,
                "platform": item["platform"],
                "source_id": item["source_id"],
                "error": error_text,
                "at": datetime.now(timezone.utc).isoformat(),
            })
            print("FAILED:", error_text[:1400])
            print("URL remains in remaining.txt. Stopping to avoid repeated requests.")
            write_lines_atomic(REMAINING_FILE, queue)
            break

except KeyboardInterrupt:
    print("\nManual interrupt received. Current URL remains checkpointed in remaining.txt.")

finally:
    write_lines_atomic(COMPLETED_FILE, sorted(completed_set))
    write_lines_atomic(REMAINING_FILE, queue)
    print("\n" + "=" * 70)
    print("SESSION SUMMARY")
    print("=" * 70)
    print("Successful GitHub pushes:", successes)
    print("Failures:", failures)
    print("Completed total:", len(read_lines(COMPLETED_FILE)))
    print("Remaining total:", len(read_lines(REMAINING_FILE)))
    print("Completed file:", COMPLETED_FILE)
    print("Remaining file:", REMAINING_FILE)
    print("Failure log:", FAILED_LOG)
    print("GitHub push log:", PUSH_LOG)
    print("=" * 70)


SHORTSLM PERSONAL FEATURE EXTRACTION
Source URLs: 310,871
Already completed (valid records): 0
Previously completed records requiring v3.0.0 / QA refresh; requeued: 18
To process: 310,871
NOTICE: More than 10,000 source URLs detected. No URL cap is applied; the session time checkpoint remains active.
ASR: enabled | PyAV: 18.1.1
OCR: enabled
Download mode: anonymous

[1/310871] instagram | -0jDBhLinR | remaining: 310,871
Existing record requires feature bundle v3.0.0 / QA refresh; re-downloading and reprocessing: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-0jDBhLinR.json
ShortsLM downloader | instagram | -0jDBhLinR
[Instagram] Setting up session
[Instagram] Extracting URL: https://www.instagram.com/reel/-0jDBhLinR/
[Instagram] -0jDBhLinR: Checking post accessibility
[Instagram] -0jDBhLinR: Downloading JSON metadata
[info] -0jDBhLinR: Downloading 1 format(s): dash-961329048293052v+dash-3613651658910867a
[download] Destination: /tmp/shortslm_instagram_-0jDBhLinR_25d

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Feature JSON saved to Drive: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-0jDBhLinR.json
Temporary media directory removed.
PUSHED: https://github.com/Arhanpg/ShortsLM-Personal-Features/blob/main/personal_pilot/features/instagram/-0jDBhLinR.json
COMMIT: 27f3addb1d85226c9b919cfed4262b16c3ffe04d

[2/310871] instagram | -6OWa5P8W8 | remaining: 310,870
Existing record requires feature bundle v3.0.0 / QA refresh; re-downloading and reprocessing: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-6OWa5P8W8.json
ShortsLM downloader | instagram | -6OWa5P8W8
[Instagram] Setting up session
[Instagram] Extracting URL: https://www.instagram.com/reel/-6OWa5P8W8/
[Instagram] -6OWa5P8W8: Checking post accessibility
[Instagram] -6OWa5P8W8: Downloading JSON metadata
[info] -6OWa5P8W8: Downloading 1 format(s): dash-6580238555375803v+dash-959709755098942a
[download] Destination: /tmp/shortslm_instagram_-6OWa5P8W8_5omffbd_/-6OWa5P8W8.fdash-6580238555375803v.mp4
[downl

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Feature JSON saved to Drive: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-6OWa5P8W8.json
Temporary media directory removed.
PUSHED: https://github.com/Arhanpg/ShortsLM-Personal-Features/blob/main/personal_pilot/features/instagram/-6OWa5P8W8.json
COMMIT: d76562efe19ba293114e3faf8c23e205ec2d036f

[3/310871] instagram | -APvPgymrm | remaining: 310,869
Existing record requires feature bundle v3.0.0 / QA refresh; re-downloading and reprocessing: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-APvPgymrm.json
ShortsLM downloader | instagram | -APvPgymrm
[Instagram] Setting up session
[Instagram] Extracting URL: https://www.instagram.com/reel/-APvPgymrm/
[Instagram] -APvPgymrm: Checking post accessibility
[Instagram] -APvPgymrm: Downloading JSON metadata
[info] -APvPgymrm: Downloading 1 format(s): dash-1210982049800831v+dash-789272696221993a
[download] Destination: /tmp/shortslm_instagram_-APvPgymrm_yd8o6hfh/-APvPgymrm.fdash-1210982049800831v.mp4
[downl

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


... (24 outputs left out)

Feature JSON saved to Drive: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-PK2WcLBhV.json
Temporary media directory removed.
PUSHED: https://github.com/Arhanpg/ShortsLM-Personal-Features/blob/main/personal_pilot/features/instagram/-PK2WcLBhV.json
COMMIT: a83b608f80ebe488d3fec67879a5ad905fb237e2

[16/310871] instagram | -RmqQlymug | remaining: 310,856
Existing record requires feature bundle v3.0.0 / QA refresh; re-downloading and reprocessing: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-RmqQlymug.json
ShortsLM downloader | instagram | -RmqQlymug
[Instagram] Setting up session
[Instagram] Extracting URL: https://www.instagram.com/reel/-RmqQlymug/
[Instagram] -RmqQlymug: Checking post accessibility
[Instagram] -RmqQlymug: Downloading JSON metadata
[info] -RmqQlymug: Downloading 1 format(s): dash-261508670179253v+dash-350312060753718a
[download] Destination: /tmp/shortslm_instagram_-RmqQlymug_evnd0252/-RmqQlymug.fdash-261508670179253v.mp4
[downlo

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Feature JSON saved to Drive: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-RmqQlymug.json
Temporary media directory removed.
PUSHED: https://github.com/Arhanpg/ShortsLM-Personal-Features/blob/main/personal_pilot/features/instagram/-RmqQlymug.json
COMMIT: 0f25a92a3b1506604f6fe1ea7cbe1d31096045d4

[17/310871] instagram | -SnDYzABbn | remaining: 310,855
Existing record requires feature bundle v3.0.0 / QA refresh; re-downloading and reprocessing: /content/drive/MyDrive/ShortsLM_Personal_Pilot/features/instagram/-SnDYzABbn.json
ShortsLM downloader | instagram | -SnDYzABbn
[Instagram] Setting up session
[Instagram] Extracting URL: https://www.instagram.com/reel/-SnDYzABbn/
[Instagram] -SnDYzABbn: Checking post accessibility
[Instagram] -SnDYzABbn: Downloading JSON metadata
[info] -SnDYzABbn: Downloading 1 format(s): dash-1065555777938046v
[download] Destination: /tmp/shortslm_instagram_-SnDYzABbn_arqosn0a/-SnDYzABbn.mp4
[download] 100% of  131.61KiB in 00:00:00 at 1.23Mi

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


## 13. Inspect progress or resume after a Colab reconnect


In [ ]:
print("COMPLETED:")
print("\n".join(read_lines(COMPLETED_FILE)) or "(none)")
print("\nREMAINING:")
print("\n".join(read_lines(REMAINING_FILE)) or "(none)")
print("\nFeature JSON files:", sum(1 for _ in FEATURES_DIR.rglob("*.json")))
